In [ ]:
import psutil
import os
import time
import pandas as pd
import sys
from IPython.display import display

sys.path.append(os.path.abspath('../src'))
from bpe import BPETokenizer
from bbpe import BBPETokenizer

pid = os.getpid()
p = psutil.Process(pid)
p.cpu_affinity([0]) 

print(f"🚀 Environment: Simulated Mobile (1 Core) | Process ID: {pid}")

models = {}
vocab_sizes = [8000, 16000, 32000, 64000, 128000]

class CodeMixedTokenizer:
    def __init__(self, primary_bpe, fallback_bpe):
        self.primary = primary_bpe
        self.fallback = fallback_bpe
        self.known_english = set([k.replace('</w>', '') for k in self.primary.vocab.keys()])
    def encode(self, text):
        words = text.split(); final_tokens = []
        for word in words:
            clean = word.lower().strip('.,!?')
            if clean in self.known_english:
                final_tokens.extend(self.primary.encode(word + " ")['tokens'])
            else:
                tokens = self.fallback.encode(word + " ")['tokens']
                final_tokens.extend([f"[MIX]_{t}" for t in tokens])
        return {"tokens": final_tokens}

print(" Loading all BPE configurations...")
for size in vocab_sizes:
    try:
        tok = BPETokenizer()
        tok.load(f'../vocabs/bpe_{size}.json')
        models[f'BPE-{size//1000}K'] = tok
    except: pass

try:
    bbpe_fallback = BBPETokenizer()
    bbpe_fallback.load('../vocabs/bpe_8000.json')
    models['Hybrid Code-Mixed'] = CodeMixedTokenizer(models['BPE-32K'], bbpe_fallback)
except: pass

# 4. Benchmarking Logic
def benchmark_edge(tokenizer_model, text):
    start_time = time.perf_counter()
    iterations = 100
    for _ in range(iterations):
        _ = tokenizer_model.encode(text)['tokens']
    end_time = time.perf_counter()
    
    avg_latency = ((end_time - start_time) * 1000) / iterations
    throughput = (len(text.split()) * iterations) / (end_time - start_time)
    mem_usage = p.memory_info().rss / (1024 * 1024)
    
    return {"Avg Latency (ms)": round(avg_latency, 3), 
            "Throughput (Words/sec)": round(throughput, 2), 
            "Memory (MB)": round(mem_usage, 2)}

test_text = """CLIMBING A LADDER
I'm climbing a ladder.
I can't see the top.
Don't know where I'm heading.
But still, I don't stop.
The ladder is sturdy.
I climb past our roof.
I have a strange feeling
This might be a goof.
The chimney's below me.
I dare not look down.
I know it's a distance
To fall to the ground.
I reach the first cloud,
And it's thicker than cream.
I say to myself,
"This is likely a dream."
Though I'm scared of heights,
My knees do not quake.
I love this so much
That I don't want to wake."""
results = []
for name, obj in models.items():
    print(f"Profiling {name}...")
    stats = benchmark_edge(obj, test_text)
    results.append({"Model": name, **stats})

comparison_df = pd.DataFrame(results)
display(comparison_df)
comparison_df.to_csv('../reports/edge_performance_table.csv', index=False)

🚀 Environment: Simulated Mobile (1 Core) | Process ID: 18104
⏳ Loading all BPE configurations...
Profiling BPE-8K...
Profiling BPE-16K...
Profiling BPE-32K...
Profiling BPE-64K...
Profiling BPE-128K...
Profiling Hybrid Code-Mixed...


,Model,Avg Latency (ms),Throughput (Words/sec),Memory (MB)
0,BPE-8K,1.538,65021.58,258.79
1,BPE-16K,2.032,49205.84,258.79
2,BPE-32K,2.073,48249.76,258.79
3,BPE-64K,1.543,64809.49,258.79
4,BPE-128K,2.858,34990.84,258.79
5,Hybrid Code-Mixed,1.908,52404.36,258.79
